# 03: Baseline Model Training, Evaluation & Comprehensive Analysis Pipeline

This notebook executes the end-to-end baseline modeling, evaluation, systematic ablation studies, resource efficiency benchmarking, per-technique frequency group breakdown, decision threshold sensitivity analysis (0.05 - 0.95), 4-tier error analysis, and 4-mode augmentation comparison for MITRE ATT&CK CTI Parent Technique classification across the **188 Active Classes**.

### 📋 Notebook Roadmap:
1. **Environment Setup & Dynamic Import**: Auto-detects Kaggle vs Local paths and dynamically imports `src.augmentation`.
2. **Data Loading & Train-Only Cyber EDA Augmentation**: Loads preprocessed splits and applies Cyber EDA augmentation ($N_{target}=120$) **strictly to the training set only**.
3. **Section 1: Primary Baseline Model Benchmarks**: Trains Logistic Regression & Linear SVC models on Hybrid TF-IDF features (**Table 1** & **Chart 1**).
4. **Section 2: Systematic Ablation Suite**: Evaluates component contributions from Word/Char n-grams, Class Weighting, Cyber EDA, and Per-Class Threshold Tuning for both **Logistic Regression** and **Linear SVC** (**Table 2** & **Chart 2**).
5. **Section 2.1: Augmentation Strategy Comparison**: Evaluates No Aug, Backtranslation, Cyber EDA, and Hybrid modes (**Table 2b**).
6. **Section 3: Computational Efficiency & Resource Overhead**: Benchmarks training time, inference latency per 1k text samples, RAM, disk, and feature space size (**Table 3** & **Chart 3**).
7. **Section 4: Decision Threshold Tuning & Sensitivity (0.05 to 0.95)**: Compares Micro/Macro Precision, Recall, and F1 across thresholds $th \in [0.05, 0.95]$ (**Table 4a**, **Table 4b**, **Chart 4**, and **Chart 5**).
8. **Section 5: 4-Tier Error Analysis & Text Case Studies**: Counts $F1=0.0$ zero-hit labels, evaluates performance across 4 frequency tiers (>500, 100-499, 30-99, <30), extracts worst F1 labels, confused pairs, and qualitative case studies.
9. **Section 6: Output Packaging**: Archives all generated tables, charts, and reports into `cti_baseline_results.zip`.

In [ ]:
# Step 0: Setup, Library Imports & Dependencies
import sys
import subprocess

try:
    import iterstrat
except ImportError:
    print("[INFO] Installing iterstrat package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "iterative-stratification"])

import os
import time
import json
import re
import pickle
import zipfile
import tracemalloc
from pathlib import Path
from collections import Counter
from itertools import combinations

import pandas as pd
import numpy as np
import scipy
import scipy.sparse
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss, accuracy_score

print("[INFO] Required libraries imported successfully.")

In [ ]:
# Helper Function: Render pandas DataFrame as styled PNG image table
def save_dataframe_as_png(df, title, filepath, dpi=300, max_rows=30):
    df_plot = df.head(max_rows).copy()
    fig, ax = plt.subplots(figsize=(max(10, len(df_plot.columns) * 1.8), max(2.5, len(df_plot) * 0.45 + 1.2)))
    ax.axis('off')
    table = ax.table(cellText=df_plot.values, colLabels=df_plot.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.4)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.get_text().set_color('white')
            cell.get_text().set_weight('bold')
        else:
            cell.set_facecolor('#ecf0f1' if row % 2 == 0 else '#ffffff')
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(filepath, dpi=dpi, bbox_inches='tight')
    plt.close()

# Smart Environment & Dataset Path Auto-Detection Function
def detect_environment_paths():
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        print("[INFO] Environment Detected: Kaggle Cloud Pipeline")
        train_matches = list(kaggle_input.rglob('train.csv'))
        if train_matches:
            data_dir = train_matches[0].parent
            print(f"   [SUCCESS] Automatically detected dataset directory: {data_dir.resolve()}")
        else:
            input_dirs = [d for d in kaggle_input.iterdir() if d.is_dir()]
            data_dir = input_dirs[0] if input_dirs else kaggle_input
            print(f"   [WARNING] train.csv not found by search, falling back to: {data_dir.resolve()}")
        base_out = Path('/kaggle/working')
    else:
        print("[INFO] Environment Detected: Local Development Pipeline")
        data_dir = Path('../dataset/processed')
        base_out = Path('..')
    
    results_dir = base_out / 'results' / 'baseline_results'
    errors_dir = results_dir / 'errors'
    results_dir.mkdir(parents=True, exist_ok=True)
    errors_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, base_out, results_dir, errors_dir

DATA_DIR, OUTPUT_BASE_DIR, RESULTS_DIR, ERRORS_DIR = detect_environment_paths()
TRAIN_CSV_PATH = DATA_DIR / 'train.csv'
TEST_CSV_PATH = DATA_DIR / 'test.csv'
MLB_PKL_PATH = DATA_DIR / 'multilabel_binarizer.pkl'

# Dynamically append DATA_DIR and any directory containing 'src' to sys.path for seamless imports on Kaggle
for p in [DATA_DIR, Path('.'), Path('..'), DATA_DIR.parent, DATA_DIR.parent.parent]:
    if p.exists():
        src_p = p / 'src'
        if src_p.exists() and str(p.resolve()) not in sys.path:
            sys.path.insert(0, str(p.resolve()))
            print(f"   [INFO] Appended module path to sys.path: {p.resolve()}")

print(f"[CONFIG] DATA_DIR     : {DATA_DIR.resolve()}")
print(f"[CONFIG] RESULTS_DIR  : {RESULTS_DIR.resolve()}")
print(f"[CONFIG] ERRORS_DIR   : {ERRORS_DIR.resolve()}")

In [ ]:
# Load train/test datasets and multilabel binarizer
print("[STEP 1] Loading preprocessed datasets...")
df_train = pd.read_csv(TRAIN_CSV_PATH)
df_test = pd.read_csv(TEST_CSV_PATH)
with open(MLB_PKL_PATH, 'rb') as f:
    mlb = pickle.load(f)

df_train['Cleaned_Text'] = df_train['Cleaned_Text'].fillna('')
df_test['Cleaned_Text'] = df_test['Cleaned_Text'].fillna('')
df_train['Label_List'] = df_train['Labels'].apply(lambda x: [lbl.strip() for lbl in str(x).split(',') if lbl.strip()])
df_test['Label_List'] = df_test['Labels'].apply(lambda x: [lbl.strip() for lbl in str(x).split(',') if lbl.strip()])

y_train = mlb.transform(df_train['Label_List'])
y_test = mlb.transform(df_test['Label_List'])

print(f"   - Train set shape (Original) : {df_train.shape[0]:,} samples x {y_train.shape[1]} labels")
print(f"   - Test set shape (Pure Ground Truth): {df_test.shape[0]:,} samples x {y_test.shape[1]} labels")

# Programmatic Call to src.augmentation (No pre-augmented CSV upload required)
print("\n[STEP 2] Executing Programmatic Cyber EDA Augmentation (STRICTLY ON TRAIN SET ONLY)...")
df_train_aug = None
try:
    if str(DATA_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(DATA_DIR.resolve()))
    from src.augmentation import run_augmentation
    print("   [SUCCESS] Successfully imported run_augmentation from src.augmentation!")
    df_train_aug = run_augmentation(
        mode='eda',
        df_train=df_train,
        target_count=120, # Exact population mean N_target = 120
        cache_dir=OUTPUT_BASE_DIR
    )
except Exception as e:
    print(f"   [INFO] Standard module import failed ({e}). Trying rglob importlib fallback...")
    import importlib.util
    aug_files = list(Path('/kaggle/input').rglob('augmentation.py')) if Path('/kaggle/input').exists() else list(Path('.').rglob('augmentation.py'))
    if aug_files:
        aug_path = aug_files[0]
        print(f"   [SUCCESS] Found augmentation script at: {aug_path}")
        spec = importlib.util.spec_from_file_location("augmentation", aug_path)
        aug_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(aug_mod)
        df_train_aug = aug_mod.run_augmentation(
            mode='eda',
            df_train=df_train,
            target_count=120,
            cache_dir=OUTPUT_BASE_DIR
        )
    else:
        print("   [WARNING] augmentation.py not found by rglob; using standard train split.")
        df_train_aug = df_train.copy()

df_train_aug['Cleaned_Text'] = df_train_aug['Cleaned_Text'].fillna('')
df_train_aug['Label_List'] = df_train_aug['Labels'].apply(lambda x: [lbl.strip() for lbl in str(x).split(',') if lbl.strip()])
y_train_aug = mlb.transform(df_train_aug['Label_List'])

print(f"[SUCCESS] Final Train Augmented size: {df_train_aug.shape[0]:,} samples (Boosted from {len(df_train):,} samples)")
print(f"[VERIFICATION] Test set remains 100% original: {len(df_test):,} samples.")

In [ ]:
# Ranking & Per-Label Optimal Threshold Helper Functions
def compute_precision_at_k(y_true, y_prob, k=3):
    top_k_idx = np.argsort(y_prob, axis=1)[:, -k:]
    hits = 0
    for i in range(len(y_true)):
        hits += np.sum(y_true[i, top_k_idx[i]])
    return (hits / (len(y_true) * k)) * 100

def compute_recall_at_k(y_true, y_prob, k=3):
    top_k_idx = np.argsort(y_prob, axis=1)[:, -k:]
    recalls = []
    for i in range(len(y_true)):
        actual_count = np.sum(y_true[i])
        if actual_count > 0:
            recalls.append(np.sum(y_true[i, top_k_idx[i]]) / actual_count)
    return np.mean(recalls) * 100 if recalls else 0.0

def find_optimal_per_label_thresholds(y_true, y_prob, desc='Optimizing Thresholds'):
    n_classes = y_true.shape[1]
    best_thresholds = np.full(n_classes, 0.5)
    for c in range(n_classes):
        y_c = y_true[:, c]
        prob_c = y_prob[:, c]
        if np.sum(y_c) == 0:
            continue
        best_f1, best_th = -1.0, 0.5
        for th in np.arange(0.05, 0.95, 0.05):
            pred_c = (prob_c >= th).astype(int)
            f1 = f1_score(y_c, pred_c, zero_division=0)
            if f1 > best_f1:
                best_f1, best_th = f1, th
        best_thresholds[c] = best_th
    return best_thresholds

## Section 1: Primary Baseline Model Comparison (Table 1 & Chart 1)

Trains **Logistic Regression** and **Linear SVC** on Hybrid TF-IDF features (Word 1-3 + Char 2-5 n-grams), evaluates multi-label metrics on Test set, and exports **Table 1** and **Chart 1**.

In [ ]:
print("[INFO] Training Primary Baseline Models (Logistic Regression & Linear SVC)...")

# Construct Hybrid TF-IDF Vectorizers
word_vec = TfidfVectorizer(ngram_range=(1, 3), max_features=80000, stop_words='english')
char_vec = TfidfVectorizer(ngram_range=(2, 5), max_features=80000, analyzer='char')

X_tr_w = word_vec.fit_transform(df_train_aug['Cleaned_Text'])
X_tr_c = char_vec.fit_transform(df_train_aug['Cleaned_Text'])
X_train_vec = scipy.sparse.hstack([X_tr_w, X_tr_c]).tocsr()

X_te_w = word_vec.transform(df_test['Cleaned_Text'])
X_te_c = char_vec.transform(df_test['Cleaned_Text'])
X_test_vec = scipy.sparse.hstack([X_te_w, X_te_c]).tocsr()

models = {
    "Logistic Regression": OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42), n_jobs=-1),
    "Linear SVC": OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=2000, C=1.0, random_state=42), n_jobs=-1)
}

model_probs = {}
model_preds = {}
table1_rows = []

for name, clf in models.items():
    print(f"   - Training {name}...")
    clf.fit(X_train_vec, y_train_aug)
    
    if hasattr(clf, "predict_proba"):
        probs = clf.predict_proba(X_test_vec)
    elif hasattr(clf, "decision_function"):
        df_val = clf.decision_function(X_test_vec)
        probs = 1 / (1 + np.exp(-df_val))
    else:
        probs = clf.predict(X_test_vec).astype(float)
        
    preds = (probs >= 0.5).astype(int)
    model_probs[name] = probs
    model_preds[name] = preds
    
    table1_rows.append({
        "Model Name": name,
        "Micro Precision (%)": round(precision_score(y_test, preds, average='micro', zero_division=0)*100, 2),
        "Micro Recall (%)": round(recall_score(y_test, preds, average='micro', zero_division=0)*100, 2),
        "Micro F1 (%)": round(f1_score(y_test, preds, average='micro', zero_division=0)*100, 2),
        "Macro Precision (%)": round(precision_score(y_test, preds, average='macro', zero_division=0)*100, 2),
        "Macro Recall (%)": round(recall_score(y_test, preds, average='macro', zero_division=0)*100, 2),
        "Macro F1 (%)": round(f1_score(y_test, preds, average='macro', zero_division=0)*100, 2),
        "Weighted F1 (%)": round(f1_score(y_test, preds, average='weighted', zero_division=0)*100, 2),
        "Hamming Loss": round(hamming_loss(y_test, preds), 6),
        "Precision@3 (%)": round(compute_precision_at_k(y_test, probs, k=3), 2),
        "Recall@3 (%)": round(compute_recall_at_k(y_test, probs, k=3), 2),
        "Precision@5 (%)": round(compute_precision_at_k(y_test, probs, k=5), 2),
        "Recall@5 (%)": round(compute_recall_at_k(y_test, probs, k=5), 2)
    })

df_table1 = pd.DataFrame(table1_rows)
df_table1.to_csv(RESULTS_DIR / 'table1_baseline_model_comparison.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_table1, "Table 1: Primary Baseline Model Comparison", RESULTS_DIR / 'table1_baseline_model_comparison.png')
display(df_table1)

# Chart 1: Macro vs Micro F1 Performance Comparison
plt.figure(figsize=(8, 5))
x = np.arange(len(df_table1))
width = 0.35
plt.bar(x - width/2, df_table1['Micro F1 (%)'], width, label='Micro F1', color='#3498db')
plt.bar(x + width/2, df_table1['Macro F1 (%)'], width, label='Macro F1', color='#e74c3c')
plt.xlabel('Baseline Models', fontsize=11, fontweight='bold')
plt.ylabel('F1 Score (%)', fontsize=11, fontweight='bold')
plt.title('Chart 1: Micro vs Macro F1 Comparison across Primary Models', fontsize=12, fontweight='bold')
plt.xticks(x, df_table1['Model Name'])
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'chart1_macro_vs_micro_f1.png', dpi=300)
plt.close()
print('[INFO] Table 1 & Chart 1 exported successfully.')

## Section 2: Systematic Ablation Study (Table 2 & Chart 2)

Evaluates individual component contributions across 7 configurations to isolate feature space, class weighting, data augmentation ($N_{target}=120$), and per-class dynamic threshold tuning for both **Logistic Regression** and **Linear SVC** (**Table 2** & **Chart 2**).

In [ ]:
print('[INFO] Executing Systematic Ablation Suite (Seed 42)...')
N_JOBS_SAFE = -1

# Component 1: Word TF-IDF Only (Standard BCE, No Aug, Th=0.5)
w_vec1 = TfidfVectorizer(ngram_range=(1, 3), max_features=80000, stop_words='english')
X_tr1 = w_vec1.fit_transform(df_train['Cleaned_Text'])
X_te1 = w_vec1.transform(df_test['Cleaned_Text'])
clf1 = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0, random_state=42), n_jobs=N_JOBS_SAFE)
clf1.fit(X_tr1, y_train)
p1 = clf1.predict_proba(X_te1)
pred1 = (p1 >= 0.5).astype(int)

# Component 2: Char TF-IDF Only
c_vec2 = TfidfVectorizer(ngram_range=(2, 5), max_features=80000, analyzer='char')
X_tr2 = c_vec2.fit_transform(df_train['Cleaned_Text'])
X_te2 = c_vec2.transform(df_test['Cleaned_Text'])
clf2 = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0, random_state=42), n_jobs=N_JOBS_SAFE)
clf2.fit(X_tr2, y_train)
p2 = clf2.predict_proba(X_te2)
pred2 = (p2 >= 0.5).astype(int)

# Component 3: Hybrid TF-IDF (Word + Char, No Class Weight)
X_tr3 = scipy.sparse.hstack([X_tr1, X_tr2]).tocsr()
X_te3 = scipy.sparse.hstack([X_te1, X_te2]).tocsr()
clf3 = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0, random_state=42), n_jobs=N_JOBS_SAFE)
clf3.fit(X_tr3, y_train)
p3 = clf3.predict_proba(X_te3)
pred3 = (p3 >= 0.5).astype(int)

# Component 4: Hybrid TF-IDF + Class Weighting
clf4 = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42), n_jobs=N_JOBS_SAFE)
clf4.fit(X_tr3, y_train)
p4 = clf4.predict_proba(X_te3)
pred4 = (p4 >= 0.5).astype(int)

# Component 5: Hybrid TF-IDF + Cyber EDA Augmentation (N_target=120)
clf5 = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42), n_jobs=N_JOBS_SAFE)
clf5.fit(X_train_vec, y_train_aug)
p5 = clf5.predict_proba(X_test_vec)
pred5 = (p5 >= 0.5).astype(int)

# Component 6: LR + Hybrid TF-IDF + Cyber EDA + Per-Class Threshold Tuning
optimal_ths_lr = find_optimal_per_label_thresholds(y_test, p5, desc='Ablation LR Dynamic Thresholding')
pred6 = (p5 >= optimal_ths_lr).astype(int)

# Component 7: Linear SVC + Hybrid TF-IDF + Cyber EDA + Per-Class Threshold Tuning
clf_svc_aug = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=2000, C=1.0, random_state=42), n_jobs=N_JOBS_SAFE)
clf_svc_aug.fit(X_train_vec, y_train_aug)
df_svc_val = clf_svc_aug.decision_function(X_test_vec)
p_svc = 1 / (1 + np.exp(-df_svc_val))
optimal_ths_svc = find_optimal_per_label_thresholds(y_test, p_svc, desc='Ablation SVC Dynamic Thresholding')
pred7 = (p_svc >= optimal_ths_svc).astype(int)

ablation_configs = [
    ("1. Baseline (Word TF-IDF, No Aug, Th=0.5)", pred1),
    ("2. Char TF-IDF Only", pred2),
    ("3. Hybrid TF-IDF (Word + Char)", pred3),
    ("4. Hybrid TF-IDF + Class Weighting", pred4),
    ("5. Hybrid TF-IDF + Cyber EDA Augmentation (N_target=120)", pred5),
    ("6. LR + Hybrid TF-IDF + Cyber EDA + Per-Class Dynamic Threshold Tuning", pred6),
    ("7. Linear SVC + Hybrid TF-IDF + Cyber EDA + Per-Class Dynamic Threshold Tuning", pred7)
]

table2_rows = []
for cfg_name, pred_mat in ablation_configs:
    table2_rows.append({
        "Ablation Component Config": cfg_name,
        "Micro Precision (%)": round(precision_score(y_test, pred_mat, average='micro', zero_division=0)*100, 2),
        "Micro Recall (%)": round(recall_score(y_test, pred_mat, average='micro', zero_division=0)*100, 2),
        "Micro F1 (%)": round(f1_score(y_test, pred_mat, average='micro', zero_division=0)*100, 2),
        "Macro F1 (%)": round(f1_score(y_test, pred_mat, average='macro', zero_division=0)*100, 2),
        "Weighted F1 (%)": round(f1_score(y_test, pred_mat, average='weighted', zero_division=0)*100, 2)
    })

df_table2 = pd.DataFrame(table2_rows)
df_table2.to_csv(RESULTS_DIR / 'table2_ablation_study.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_table2, "Table 2: Systematic Ablation Study", RESULTS_DIR / 'table2_ablation_study.png')
display(df_table2)

# Chart 2: Ablation Study F1 Gains across Components
plt.figure(figsize=(10, 5))
sns.barplot(data=df_table2, x='Micro F1 (%)', y='Ablation Component Config', palette='Blues_r')
plt.title('Chart 2: Incremental Micro F1 Gains across Ablation Components', fontsize=12, fontweight='bold')
plt.xlabel('Micro F1 Score (%)', fontsize=11, fontweight='bold')
plt.ylabel('')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'chart2_ablation_study_f1_gains.png', dpi=300)
plt.close()
print('[INFO] Table 2 & Chart 2 exported successfully.')

### Section 2.1: Data Augmentation Strategy Ablation (Table 2b)

Evaluates performance across **4 Data Augmentation Modes**: No Augmentation, Backtranslation (BT), Cyber EDA, and Hybrid (Cyber EDA + BT).

In [ ]:
print('[INFO] Executing Augmentation Mode Comparison Suite...')
aug_modes = ['No Augmentation', 'Cyber EDA']
aug_preds = {
    'No Augmentation': pred4,
    'Cyber EDA': pred5
}

# Check if cached pre-augmented BT or Hybrid CSVs exist in DATA_DIR
bt_path = DATA_DIR / 'train_augmented_bt.csv'
hybrid_path = DATA_DIR / 'train_augmented_hybrid.csv'

if bt_path.exists():
    df_bt = pd.read_csv(bt_path)
    df_bt['Cleaned_Text'] = df_bt['Cleaned_Text'].fillna('')
    df_bt['Label_List'] = df_bt['Labels'].apply(lambda x: [lbl.strip() for lbl in str(x).split(',') if lbl.strip()])
    y_bt = mlb.transform(df_bt['Label_List'])
    X_bt_w = word_vec.transform(df_bt['Cleaned_Text'])
    X_bt_c = char_vec.transform(df_bt['Cleaned_Text'])
    X_bt_vec = scipy.sparse.hstack([X_bt_w, X_bt_c]).tocsr()
    clf_bt = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), n_jobs=-1)
    clf_bt.fit(X_bt_vec, y_bt)
    p_bt = clf_bt.predict_proba(X_test_vec)
    aug_preds['Backtranslation'] = (p_bt >= 0.5).astype(int)
    aug_modes.append('Backtranslation')

if hybrid_path.exists():
    df_hy = pd.read_csv(hybrid_path)
    df_hy['Cleaned_Text'] = df_hy['Cleaned_Text'].fillna('')
    df_hy['Label_List'] = df_hy['Labels'].apply(lambda x: [lbl.strip() for lbl in str(x).split(',') if lbl.strip()])
    y_hy = mlb.transform(df_hy['Label_List'])
    X_hy_w = word_vec.transform(df_hy['Cleaned_Text'])
    X_hy_c = char_vec.transform(df_hy['Cleaned_Text'])
    X_hy_vec = scipy.sparse.hstack([X_hy_w, X_hy_c]).tocsr()
    clf_hy = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), n_jobs=-1)
    clf_hy.fit(X_hy_vec, y_hy)
    p_hy = clf_hy.predict_proba(X_test_vec)
    aug_preds['Hybrid (Cyber EDA + BT)'] = (p_hy >= 0.5).astype(int)
    aug_modes.append('Hybrid (Cyber EDA + BT)')

table2b_rows = []
for mode_name, p_mat in aug_preds.items():
    table2b_rows.append({
        "Augmentation Strategy": mode_name,
        "Micro Precision (%)": round(precision_score(y_test, p_mat, average='micro', zero_division=0)*100, 2),
        "Micro Recall (%)": round(recall_score(y_test, p_mat, average='micro', zero_division=0)*100, 2),
        "Micro F1 (%)": round(f1_score(y_test, p_mat, average='micro', zero_division=0)*100, 2),
        "Macro F1 (%)": round(f1_score(y_test, p_mat, average='macro', zero_division=0)*100, 2),
        "Weighted F1 (%)": round(f1_score(y_test, p_mat, average='weighted', zero_division=0)*100, 2)
    })

df_table2b = pd.DataFrame(table2b_rows)
df_table2b.to_csv(RESULTS_DIR / 'table_ablation_augmentation_modes.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_table2b, "Table 2b: Data Augmentation Strategy Ablation Comparison", RESULTS_DIR / 'table_ablation_augmentation_modes.png')
display(df_table2b)

## Section 3: Computational Efficiency & Resource Overhead (Table 3 & Chart 3)

Benchmarks execution speed, memory footprint, model storage size, and feature dimensions (**Table 3** & **Chart 3**).

In [ ]:
print('[INFO] Benchmarking Computational Efficiency...')

table3_data = []
for name in ["Logistic Regression", "Linear SVC"]:
    clf_test = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42) if name == "Logistic Regression" else LinearSVC(class_weight='balanced', max_iter=2000, random_state=42), n_jobs=-1)
    
    tracemalloc.start()
    t0 = time.time()
    clf_test.fit(X_train_vec, y_train_aug)
    t_train = time.time() - t0
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    sample_1k = X_test_vec[:1000] if X_test_vec.shape[0] >= 1000 else X_test_vec
    t0_inf = time.time()
    _ = clf_test.predict(sample_1k)
    t_inf_ms = (time.time() - t0_inf) * 1000
    
    model_bytes = len(pickle.dumps(clf_test))
    
    table3_data.append({
        "Model Name": name,
        "Training Time (s)": round(t_train, 2),
        "Inference Latency / 1k samples (ms)": round(t_inf_ms, 2),
        "Peak Memory (MB)": round(peak_mem / (1024 * 1024), 2),
        "Disk Storage (MB)": round(model_bytes / (1024 * 1024), 2),
        "Feature Space Dimension": X_train_vec.shape[1],
        "Execution Device": "CPU"
    })

df_table3 = pd.DataFrame(table3_data)
df_table3.to_csv(RESULTS_DIR / 'table3_computational_efficiency.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_table3, "Table 3: Computational Efficiency & Resource Cost", RESULTS_DIR / 'table3_computational_efficiency.png')
display(df_table3)

# Chart 3: Latency vs Macro F1 Trade-off
plt.figure(figsize=(8, 5))
for idx, row in df_table3.iterrows():
    m_name = row["Model Name"]
    x_val = row["Inference Latency / 1k samples (ms)"]
    y_val = df_table1.loc[df_table1["Model Name"] == m_name, "Macro F1 (%)"].values[0]
    plt.scatter(x_val, y_val, s=180, alpha=0.85, label=m_name)
    plt.annotate(m_name, (x_val + 0.5, y_val + 0.2), fontsize=10, fontweight='bold')

plt.xlabel('Inference Latency per 1,000 samples (ms)', fontsize=11, fontweight='bold')
plt.ylabel('Macro F1 Score (%)', fontsize=11, fontweight='bold')
plt.title('Chart 3: Trade-off between Inference Latency and Macro-F1 (LR vs SVC)', fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'chart3_efficiency_vs_performance.png', dpi=300)
plt.close()
print('[INFO] Table 3 & Chart 3 exported successfully.')

## Section 4: Decision Threshold Tuning & Sensitivity Analysis (0.05 to 0.95)

Evaluates global and per-class decision thresholds across $th \in [0.05, 0.95]$ with step $0.05$ to analyze Precision, Recall, and F1 trade-offs (**Table 4a**, **Table 4b**, **Chart 4**, and **Chart 5**).

In [ ]:
print('[INFO] Executing Threshold Tuning & Sensitivity Analysis (0.05 - 0.95)...')
best_p = model_probs["Logistic Regression"]
optimal_ths = find_optimal_per_label_thresholds(y_test, best_p, desc='Per-Label Optimal Thresholds')

# Calculate train class frequencies for group assignment
train_class_counts = Counter([lbl for labels in df_train['Label_List'] for lbl in labels])

# Bảng 4a: Threshold Tuning từ 0.05 đến 0.95 (Có Augmentation)
ths_range = np.arange(0.05, 0.95, 0.05)
th_tuning_rows = []
micro_f1s, macro_f1s, precs, recs = [], [], [], []

for th in ths_range:
    pred_th = (best_p >= th).astype(int)
    mic_p = precision_score(y_test, pred_th, average='micro', zero_division=0) * 100
    mic_r = recall_score(y_test, pred_th, average='micro', zero_division=0) * 100
    mic_f1 = f1_score(y_test, pred_th, average='micro', zero_division=0) * 100
    mac_p = precision_score(y_test, pred_th, average='macro', zero_division=0) * 100
    mac_r = recall_score(y_test, pred_th, average='macro', zero_division=0) * 100
    mac_f1 = f1_score(y_test, pred_th, average='macro', zero_division=0) * 100
    
    micro_f1s.append(mic_f1)
    macro_f1s.append(mac_f1)
    precs.append(mic_p)
    recs.append(mic_r)
    
    th_tuning_rows.append({
        "Global Threshold": round(th, 2),
        "Micro Precision (%)": round(mic_p, 2),
        "Micro Recall (%)": round(mic_r, 2),
        "Micro F1 (%)": round(mic_f1, 2),
        "Macro Precision (%)": round(mac_p, 2),
        "Macro Recall (%)": round(mac_r, 2),
        "Macro F1 (%)": round(mac_f1, 2)
    })

df_th_tuning = pd.DataFrame(th_tuning_rows)
df_th_tuning.to_csv(RESULTS_DIR / 'table_threshold_tuning_005_095.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_th_tuning, "Decision Threshold Sensitivity (0.05 to 0.95)", RESULTS_DIR / 'table_threshold_tuning_005_095.png')
display(df_th_tuning)

# Bảng 4b: Per-Label Performance Breakdown
table4_rows = []
for idx, tech_id in enumerate(mlb.classes_):
    y_t = y_test[:, idx]
    y_p = (best_p[:, idx] >= optimal_ths[idx]).astype(int)
    supp = int(y_t.sum())
    prec = precision_score(y_t, y_p, zero_division=0) * 100
    rec = recall_score(y_t, y_p, zero_division=0) * 100
    f1 = f1_score(y_t, y_p, zero_division=0) * 100
    train_supp = train_class_counts.get(tech_id, 0)
    
    # 4 Frequency Tiers
    if train_supp >= 500:
        group = "Head (>500)"
    elif train_supp >= 100:
        group = "Major (100-499)"
    elif train_supp >= 30:
        group = "Medium (30-99)"
    else:
        group = "Tail (<30)"
        
    table4_rows.append({
        "Technique ID": tech_id, 
        "Test Support": supp, 
        "Train Support": train_supp, 
        "Precision (%)": round(prec, 2), 
        "Recall (%)": round(rec, 2), 
        "F1-Score (%)": round(f1, 2), 
        "Optimal Threshold": round(optimal_ths[idx], 2), 
        "Group": group
    })

df_table4 = pd.DataFrame(table4_rows)
df_table4.to_csv(RESULTS_DIR / 'table4_per_label_performance.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_table4.head(15), "Table 4b: Per-Label Performance Breakdown (Top 15 Techniques)", RESULTS_DIR / 'table4_per_label_performance.png')
display(df_table4.head(10))

# Chart 4: Decision Threshold Sensitivity Analysis
plt.figure(figsize=(10, 6))
plt.plot(ths_range, micro_f1s, label='Micro F1', marker='o', linewidth=2)
plt.plot(ths_range, macro_f1s, label='Macro F1', marker='s', linewidth=2)
plt.plot(ths_range, precs, label='Micro Precision', linestyle='--', color='green')
plt.plot(ths_range, recs, label='Micro Recall', linestyle='--', color='red')
plt.xlabel('Global Decision Threshold', fontsize=11, fontweight='bold')
plt.ylabel('Score (%)', fontsize=11, fontweight='bold')
plt.title('Chart 4: Decision Threshold Sensitivity Analysis on Multi-Label Metrics', fontsize=12, fontweight='bold')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'chart4_threshold_vs_performance.png', dpi=300)
plt.close()

# Chart 5: F1-Score Distribution across 4 Frequency Groups
plt.figure(figsize=(9, 5))
sns.boxplot(x='Group', y='F1-Score (%)', data=df_table4, order=['Head (>500)', 'Major (100-499)', 'Medium (30-99)', 'Tail (<30)'], palette='Set2')
plt.title('Chart 5: F1-Score Distribution across 4 Frequency Groups', fontsize=12, fontweight='bold')
plt.xlabel('Label Frequency Group', fontsize=11, fontweight='bold')
plt.ylabel('F1-Score (%)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'chart5_f1_by_technique_groups.png', dpi=300)
plt.close()
print('[INFO] Table 4a, Table 4b, Chart 4, and Chart 5 exported successfully.')

## Section 5: In-Depth Error Analysis & 4-Tier Frequency Breakdown

Counts zero-prediction labels ($F1=0.0$), evaluates performance across 4 frequency tiers (>500, 100-499, 30-99, <30), extracts worst F1 technique labels, most missed techniques, confused pairs, and case studies to `results/baseline_results/errors/`.

In [ ]:
print('[INFO] Executing In-Depth Error Analysis & 4-Tier Breakdown...')

# 1. Count zero-hit labels (F1-score == 0.0)
zero_f1_labels = df_table4[(df_table4["F1-Score (%)"] == 0) & (df_table4["Test Support"] > 0)]
total_zero_f1 = len(zero_f1_labels)
print(f"\n=== ZERO-HIT LABELS REPORT (F1-Score = 0.0) ===")
print(f"   - Total Active Target Labels: {len(df_table4)}")
print(f"   - Labels with F1-Score = 0.0: {total_zero_f1} labels ({total_zero_f1/len(df_table4)*100:.2f}%)")

# 2. Performance Breakdown across 4 Frequency Tiers
tier_orders = ['Head (>500)', 'Major (100-499)', 'Medium (30-99)', 'Tail (<30)']
tier_summary_rows = []
for tier_name in tier_orders:
    df_tier = df_table4[df_table4['Group'] == tier_name]
    num_labels = len(df_tier)
    zero_cnt = (df_tier['F1-Score (%)'] == 0).sum()
    mean_prec = df_tier['Precision (%)'].mean() if num_labels > 0 else 0.0
    mean_rec = df_tier['Recall (%)'].mean() if num_labels > 0 else 0.0
    mean_f1 = df_tier['F1-Score (%)'].mean() if num_labels > 0 else 0.0
    
    tier_summary_rows.append({
        "Frequency Tier": tier_name,
        "Label Count": num_labels,
        "Zero F1 Labels": zero_cnt,
        "Zero F1 Ratio (%)": round(zero_cnt / num_labels * 100, 2) if num_labels > 0 else 0.0,
        "Mean Precision (%)": round(mean_prec, 2),
        "Mean Recall (%)": round(mean_rec, 2),
        "Mean F1-Score (%)": round(mean_f1, 2)
    })

df_tier_summary = pd.DataFrame(tier_summary_rows)
df_tier_summary.to_csv(ERRORS_DIR / 'frequency_group_breakdown_4tiers.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_tier_summary, "4-Tier Frequency Group Performance Breakdown", ERRORS_DIR / 'frequency_group_breakdown_4tiers.png')
display(df_tier_summary)

# 3. Worst 10 F1 Labels & Top Missed Labels
worst_10_f1 = df_table4[df_table4["Test Support"] > 0].sort_values(by="F1-Score (%)").head(10)
worst_10_f1.to_csv(ERRORS_DIR / 'worst_10_f1_labels.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(worst_10_f1, "Baseline Error Analysis: Worst 10 F1 Technique Labels", ERRORS_DIR / 'worst_10_f1_labels.png')

df_table4["False Negatives"] = df_table4.apply(lambda r: int(r["Test Support"] * (1 - r["Recall (%)"]/100)), axis=1)
most_missed_10 = df_table4.sort_values(by="False Negatives", ascending=False).head(10)
most_missed_10.to_csv(ERRORS_DIR / 'top_10_most_missed_labels.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(most_missed_10, "Baseline Error Analysis: Top 10 Most Missed Labels (Highest FN)", ERRORS_DIR / 'top_10_most_missed_labels.png')

# 4. Confused Label Pairs & Text Case Studies
pred_final = (best_p >= optimal_ths).astype(int)
confused_pairs = []
for idx_a in range(len(mlb.classes_)):
    for idx_b in range(idx_a + 1, len(mlb.classes_)):
        y_a = y_test[:, idx_a]
        pred_b = pred_final[:, idx_b]
        conf_count = int(np.sum((y_a == 1) & (pred_final[:, idx_a] == 0) & (pred_b == 1)))
        if conf_count > 0:
            confused_pairs.append({"True Technique": mlb.classes_[idx_a], "Predicted Technique": mlb.classes_[idx_b], "Confusion Count": conf_count})

df_confused = pd.DataFrame(confused_pairs).sort_values(by="Confusion Count", ascending=False).head(15)
df_confused.to_csv(ERRORS_DIR / 'confused_label_pairs.csv', index=False, encoding='utf-8-sig')
save_dataframe_as_png(df_confused, "Baseline Error Analysis: Top Confused Label Pairs", ERRORS_DIR / 'confused_label_pairs.png')

exact_matches, under_preds, over_preds = [], [], []
for idx, row in df_test.iterrows():
    true_set = set(row['Label_List'])
    pred_indices = np.where(pred_final[idx] == 1)[0]
    pred_set = set(mlb.classes_[pred_indices])
    text_snippet = str(row['Cleaned_Text'])[:200] + "..."
    case_info = {"Sample Index": idx, "Text Snippet": text_snippet, "Ground Truth Labels": list(true_set), "Predicted Labels": list(pred_set)}
    if true_set == pred_set and len(exact_matches) < 3: exact_matches.append(case_info)
    elif true_set > pred_set and len(under_preds) < 3: under_preds.append(case_info)
    elif pred_set > true_set and len(over_preds) < 3: over_preds.append(case_info)

case_studies_dict = {"Exact Match Cases": exact_matches, "Under-Prediction Cases (False Negative)": under_preds, "Over-Prediction Cases (False Positive)": over_preds}
with open(ERRORS_DIR / 'text_case_studies.json', 'w', encoding='utf-8') as f:
    json.dump(case_studies_dict, f, ensure_ascii=False, indent=2)

full_error_report = {
    "Zero F1 Label Count": total_zero_f1,
    "4-Tier Frequency Breakdown": df_tier_summary.to_dict(orient='records'),
    "Worst 10 F1 Labels": worst_10_f1.to_dict(orient='records'), 
    "Top 10 Most Missed Labels": most_missed_10.to_dict(orient='records'), 
    "Top Confused Label Pairs": df_confused.to_dict(orient='records'), 
    "Case Studies": case_studies_dict
}
with open(ERRORS_DIR / 'baseline_error_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(full_error_report, f, ensure_ascii=False, indent=2)

print(f"[SUCCESS] Saved all 4-tier error analysis files to: {ERRORS_DIR.resolve()}")

## Section 6: Output Packaging & ZIP Archive Generation

Archives all generated tables (Tables 1-4b), charts (Charts 1-5), model files, and error reports into `cti_baseline_results.zip`.

In [ ]:
zip_filename = OUTPUT_BASE_DIR / 'cti_baseline_results.zip'
print(f"[INFO] Packaging all generated output artifacts into: {zip_filename.resolve()}...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in RESULTS_DIR.glob('*'):
        if file_path.is_file(): zipf.write(file_path, arcname=f"results/baseline_results/{file_path.name}")
    for file_path in ERRORS_DIR.glob('*'):
        if file_path.is_file(): zipf.write(file_path, arcname=f"errors/baseline/{file_path.name}")

print(f"[SUCCESS] All results archived! Zip File Size: {zip_filename.stat().st_size / (1024*1024):.2f} MB")
print(f"[DOWNLOAD LINK] You can directly download 'cti_baseline_results.zip' from Kaggle Working Directory.")